# T05 — LIANA: Cell-Cell Communication

**Tool:** `liana-py` (scverse core 2025)  
**Dataset:** PBMC 3k (processed from T00)  
**Paper:** [Türei et al. 2024, Nature Cell Biology](https://doi.org/10.1038/s41556-024-01422-7)

---

## What is cell-cell communication (CCC)?

Cells don't act in isolation — they signal to each other through ligand-receptor (LR) interactions. A ligand secreted by cell type A binds a receptor on cell type B, triggering a downstream pathway. Mapping these interactions tells you:
- Which cell types are "talking" and "listening"
- Which signaling pathways are active in a tissue/condition
- How perturbations or disease alter inter-cellular signaling

## Why LIANA instead of just squidpy's `ligrec`?

Squidpy's `sq.gr.ligrec` runs *one* LR method (permutation test with one database). The field has 7+ competing methods (CellChat, CellPhoneDB, NicheNet, Connectome, NATMI, etc.), each with different statistical assumptions and databases. They often disagree.

**LIANA** (LIgand-receptor ANAlysis framework) solves this by:
1. Running **multiple methods simultaneously** on the same data
2. Providing a **consensus score** that ranks LR pairs by agreement across methods
3. Unifying **multiple LR databases** (CellChatDB, CellPhoneDB, NATMI, Consensus)
4. Supporting **downstream pathway analysis** of active signaling

```
adata (clustered) → LIANA → per-cluster-pair × LR-interaction scores
                                ↓
                      consensus ranking + visualization
```

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import liana
import matplotlib.pyplot as plt

sc.settings.set_figure_params(dpi=80, facecolor='white')
print(f'liana {liana.__version__}')

## 1. Load Processed Data

LIANA needs:
- A clustered/annotated AnnData with `adata.obs[groupby_key]`
- Raw or normalized counts (method-dependent; LIANA handles this internally)
- Ideally `adata.raw` preserved (some methods use raw counts)

In [ ]:
import os
if os.path.exists('../data/pbmc3k_processed.h5ad'):
    adata = sc.read_h5ad('../data/pbmc3k_processed.h5ad')
else:
    # Rebuild PBMC 3k with cell type labels
    adata = sc.datasets.pbmc3k_processed()
    # pbmc3k_processed() already has louvain clusters; rename for consistency
    if 'louvain' in adata.obs and 'cell_type' not in adata.obs:
        adata.obs['cell_type'] = adata.obs['louvain']

print(adata)
print('\nCell type distribution:')
print(adata.obs['cell_type'].value_counts())

## 2. LIANA: Running Multiple Methods at Once

The core API: `liana.mt.rank_aggregate` runs a set of methods and produces a **consensus rank** per LR pair per cell-type pair.

Available methods in LIANA:
- `natmi` — expression product of ligand × receptor
- `connectome` — scaled expression product
- `logfc` — log-fold change of LR vs. background
- `singlecellsignalr` — LRscore from SingleCellSignalR
- `cellphonedb` — permutation test (mean)
- `geometric_mean` — geometric mean of expressions
- `rank_aggregate` — **consensus**: aggregate rank across all above

In [ ]:
# Run all methods + produce consensus aggregate rank
# groupby: the obs column with cell type labels
# resource_name: which LR database to use
#   options: 'consensus' (union of CellChatDB, CellPhoneDB, NATMI, etc.)
#            'cellchatdb', 'cellphonedb', 'lrdb', ...

liana.mt.rank_aggregate(
    adata,
    groupby='cell_type',
    resource_name='consensus',   # recommended: large curated union
    expr_prop=0.1,               # min fraction of cells expressing gene
    verbose=True,
    use_raw=False,
    n_perms=100,                 # permutations for cellphonedb; 1000 for publication
    seed=42,
)

# Results stored in adata.uns['liana_res']
print('LIANA result columns:', adata.uns['liana_res'].columns.tolist())
print('Shape:', adata.uns['liana_res'].shape)

In [ ]:
# Inspect the top consensus interactions
# aggregate_rank: low = high-confidence interaction (rank from 0→1)
result = adata.uns['liana_res']
print(result.sort_values('aggregate_rank').head(20)[
    ['source', 'target', 'ligand_complex', 'receptor_complex', 'aggregate_rank']
])

## 3. Visualization

LIANA has a rich plotting API: dotplots, chord diagrams, heatmaps.

In [ ]:
# Dotplot: top interactions across all cell type pairs
# size = expr_prod (co-expression), color = aggregate_rank (consensus confidence)
liana.pl.dotplot(
    adata,
    colour='aggregate_rank',
    size='lr_means',
    inverse_colour=True,      # low rank = dark = good
    source_labels=['CD14+ Mono', 'DC'],
    target_labels=['CD4+ T', 'NK', 'B cell'],
    top_n=20,
    orderby='aggregate_rank',
    orderby_ascending=True,
    figure_size=(10, 6),
)

In [ ]:
# Heatmap: total communication strength between each cell type pair
# Shows which pairs have the most active signaling
liana.pl.heat_stable(
    adata.uns['liana_res'],
    score_col='aggregate_rank',
    inverse_score=True,
    title='CCC magnitude (LIANA consensus)',
)

In [ ]:
# Chord diagram: directionality of communication
# (requires circlify or chord; may need: pip install chord)
try:
    liana.pl.circle_plot(
        adata.uns['liana_res'],
        score_col='aggregate_rank',
        inverse_score=True,
    )
except Exception as e:
    print(f'Chord plot unavailable: {e}')
    print('Install with: pip install chord circlify')

## 4. Accessing Individual Method Scores

Beyond the consensus, you can inspect each method's score independently.

In [ ]:
# Individual method scores are columns in the result DataFrame
method_cols = [c for c in result.columns if c not in
               ['source', 'target', 'ligand', 'receptor',
                'ligand_complex', 'receptor_complex', 'aggregate_rank']]
print('Per-method score columns:', method_cols)

# Focus on a specific sender → receiver pair
mono_to_t = result[
    (result['source'] == 'CD14+ Mono') &
    (result['target'] == 'CD4+ T')
].sort_values('aggregate_rank').head(10)
print('\nTop Monocyte → CD4+ T interactions:')
print(mono_to_t[['ligand_complex', 'receptor_complex', 'aggregate_rank', 'lr_means']])

## 5. Downstream: Linking CCC to Pathways

LIANA can connect LR interactions to intracellular signaling pathways using **CORNETO** or **decoupler-py** (connecting LIANA output to PROGENy/CollecTRI).

In [ ]:
# Filter to high-confidence interactions (aggregate_rank < 0.05)
high_conf = result[result['aggregate_rank'] < 0.05].copy()
print(f'High-confidence interactions: {len(high_conf)}')
print('\nTop sender cell types:')
print(high_conf['source'].value_counts().head())
print('\nTop ligands:')
print(high_conf['ligand_complex'].value_counts().head(10))

In [ ]:
# Available LR databases in LIANA
resources = liana.resource.select_resource('consensus')
print(f'Consensus DB: {len(resources)} LR pairs')
print(resources.head())

---
## Summary

```python
# Core LIANA workflow
liana.mt.rank_aggregate(
    adata,
    groupby='cell_type',          # obs column with cluster labels
    resource_name='consensus',    # LR database
    expr_prop=0.1,                # min expression fraction filter
)
# Result in adata.uns['liana_res'] — DataFrame with per-method + aggregate scores
```

| Score | Direction | Interpretation |
|-------|-----------|----------------|
| `aggregate_rank` | 0 = best | Consensus rank across all methods |
| `lr_means` | high = good | Mean expression of ligand × receptor |
| `cellphonedb.pvalue` | low = good | Permutation test p-value |
| `natmi.expr_prod` | high = good | Expression product |

**Key databases:** `'consensus'` (default, largest), `'cellchatdb'`, `'cellphonedb'`, `'lrdb'`

**Next:** T06 — cell2location for spatial deconvolution.